In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DATA = Path("../data")
df = pd.read_csv(DATA / "merged_school_dataset.csv")

numeric_features = [
    "fsm_pct_2024", "imd_score", "income_score", "employment_score",
    "health_score", "crime_score", "barriers_score", "living_env_score",
    "idaci_score", "adult_skills_score", "geo_barriers_score",
    "wider_barriers_score", "indoors_score", "outdoors_score",
    "p8_2023", "a8_2024",
]

df_p8 = df.loc[df["p8_2024"].notna()].copy()
X_raw = df_p8[numeric_features]
y_raw = df_p8["p8_2024"].values

X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y_raw, test_size=0.30, random_state=SEED)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=SEED)

imputer = SimpleImputer(strategy="median")
scaler  = StandardScaler()
X_train = scaler.fit_transform(imputer.fit_transform(X_train_raw))
X_val   = scaler.transform(imputer.transform(X_val_raw))
X_test  = scaler.transform(imputer.transform(X_test_raw))

y_mean, y_std = y_train.mean(), y_train.std()
y_train_scaled = (y_train - y_mean) / y_std
y_val_scaled   = (y_val   - y_mean) / y_std
y_test_scaled  = (y_test  - y_mean) / y_std

X_train_t = torch.FloatTensor(X_train)
X_val_t   = torch.FloatTensor(X_val)
X_test_t  = torch.FloatTensor(X_test)
y_train_t = torch.FloatTensor(y_train_scaled).unsqueeze(1)
y_val_t   = torch.FloatTensor(y_val_scaled).unsqueeze(1)
y_test_t  = torch.FloatTensor(y_test_scaled).unsqueeze(1)

print(f"P8 Train: {X_train_t.shape} | Val: {X_val_t.shape} | Test: {X_test_t.shape}")
print(f"Target p8_2024: train mean {y_mean:.3f}, std {y_std:.3f}")

P8 Train: torch.Size([2198, 16]) | Val: torch.Size([471, 16]) | Test: torch.Size([472, 16])
Target p8_2024: train mean 0.003, std 0.508


In [2]:
class P8ANN(nn.Module):
    def __init__(self, n_features=16, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32),         nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(32, 1),
        )
    def forward(self, x): return self.net(x)

torch.manual_seed(SEED)
model = P8ANN()
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

P8ANN(
  (net): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)
Parameters: 3,201


In [3]:
torch.manual_seed(SEED)
model = P8ANN()

EPOCHS = 300
LR = 0.001
PATIENCE = 40

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_rmse = float("inf")
best_epoch = 0
best_state = None
epochs_no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    loss = criterion(model(X_train_t), y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t).numpy().ravel() * y_std + y_mean
        val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        best_epoch = epoch
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epoch % 20 == 0:
        print(f"Epoch {epoch:>3}: val_RMSE={val_rmse:.3f}")
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

model.load_state_dict(best_state)
print(f"\nBest: epoch {best_epoch}, val RMSE={best_val_rmse:.3f}")

Epoch   0: val_RMSE=0.539
Epoch  20: val_RMSE=0.413
Epoch  40: val_RMSE=0.273
Epoch  60: val_RMSE=0.232
Epoch  80: val_RMSE=0.215
Epoch 100: val_RMSE=0.209
Epoch 120: val_RMSE=0.205
Epoch 140: val_RMSE=0.200
Epoch 160: val_RMSE=0.198
Epoch 180: val_RMSE=0.195
Epoch 200: val_RMSE=0.192
Epoch 220: val_RMSE=0.191
Epoch 240: val_RMSE=0.189
Epoch 260: val_RMSE=0.188
Epoch 280: val_RMSE=0.188

Best: epoch 286, val RMSE=0.187


In [4]:
model.eval()
with torch.no_grad():
    test_pred = model(X_test_t).numpy().ravel() * y_std + y_mean

test_r2 = r2_score(y_test, test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_mae = mean_absolute_error(y_test, test_pred)

print(f"═══ P8 ANN Test Results ═══")
print(f"R²:    {test_r2:.3f}")
print(f"RMSE:  {test_rmse:.3f}  (target SD ≈ {y_std:.3f})")
print(f"MAE:   {test_mae:.3f}")

import os
os.makedirs("../models", exist_ok=True)
torch.save(best_state, "../models/p8_ann.pt")

# RF baseline for comparison
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
r2_rf = r2_score(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))

print(f"\nP8 RF Baseline:")
print(f"R²:    {r2_rf:.3f}")
print(f"RMSE:  {rmse_rf:.3f}")

═══ P8 ANN Test Results ═══
R²:    0.894
RMSE:  0.186  (target SD ≈ 0.508)
MAE:   0.146

P8 RF Baseline:
R²:    0.868
RMSE:  0.208


In [5]:
# Permutation importance to see which features actually drive predictions
baseline_rmse = np.sqrt(mean_squared_error(y_test, test_pred))

importances = {}
for i, feat in enumerate(numeric_features):
    X_test_shuffled = X_test.copy()
    rng = np.random.default_rng(42)
    rng.shuffle(X_test_shuffled[:, i])
    with torch.no_grad():
        pred_shuffled = model(torch.FloatTensor(X_test_shuffled)).numpy().ravel()
        pred_shuffled = pred_shuffled * y_std + y_mean
    rmse_shuffled = np.sqrt(mean_squared_error(y_test, pred_shuffled))
    importances[feat] = rmse_shuffled - baseline_rmse

importance_df = pd.DataFrame({
    "feature": list(importances.keys()),
    "rmse_increase_when_shuffled": list(importances.values()),
}).sort_values("rmse_increase_when_shuffled", ascending=False)

print("P8 Permutation importance (higher = more important to predictions):")
print(importance_df.to_string(index=False))

P8 Permutation importance (higher = more important to predictions):
             feature  rmse_increase_when_shuffled
             a8_2024                     0.292809
             p8_2023                     0.253466
        fsm_pct_2024                     0.012864
       indoors_score                     0.009846
    living_env_score                     0.009432
         idaci_score                     0.006069
wider_barriers_score                     0.005324
  adult_skills_score                     0.004021
    employment_score                     0.004003
  geo_barriers_score                     0.003335
        income_score                     0.003268
      barriers_score                     0.002848
           imd_score                     0.001547
        health_score                     0.001020
         crime_score                     0.000424
      outdoors_score                    -0.000594


In [6]:
# Ablation: P8 ANN without prior-year performance
socio_features = [f for f in numeric_features if f not in ("p8_2023", "a8_2024")]

X_train_socio_raw = X_train_raw[socio_features]
X_test_socio_raw  = X_test_raw[socio_features]

imputer_s = SimpleImputer(strategy="median")
scaler_s  = StandardScaler()
X_train_s = scaler_s.fit_transform(imputer_s.fit_transform(X_train_socio_raw))
X_test_s  = scaler_s.transform(imputer_s.transform(X_test_socio_raw))

X_train_st = torch.FloatTensor(X_train_s)
X_test_st  = torch.FloatTensor(X_test_s)

torch.manual_seed(SEED)
model_socio = P8ANN(n_features=len(socio_features), dropout=0.3)
optimizer_s = torch.optim.Adam(model_socio.parameters(), lr=0.001)
criterion_s = nn.MSELoss()

for epoch in range(200):
    model_socio.train()
    optimizer_s.zero_grad()
    loss = criterion_s(model_socio(X_train_st), y_train_t)
    loss.backward()
    optimizer_s.step()

model_socio.eval()
with torch.no_grad():
    pred_socio = model_socio(X_test_st).numpy().ravel() * y_std + y_mean

r2_socio = r2_score(y_test, pred_socio)
rmse_socio = np.sqrt(mean_squared_error(y_test, pred_socio))

print(f"\nP8 ANN — SOCIO-ECONOMIC ONLY (ablation)")
print(f"R²:    {r2_socio:.3f}")
print(f"RMSE:  {rmse_socio:.3f}")
print(f"\nGap vs full model:")
print(f"  R² drop:  {0.894 - r2_socio:.3f}  (attributable to prior-year performance)")


P8 ANN — SOCIO-ECONOMIC ONLY (ablation)
R²:    0.322
RMSE:  0.470

Gap vs full model:
  R² drop:  0.572  (attributable to prior-year performance)
